In [ ]:
# =============================================================================
# Hierarchical-SAT-MPNN -- Setup
# =============================================================================
# Pipeline:
#   Stage 1 : SAT encoder, exactly as in the paper (MPNN extractor + edge attrs)
#   Stage 2 : Build chunks via balanced BFS partitioning + macro-GCN
#   Stage 3 : Cross-attention from nodes to chunks + gated fusion (Ext-3 style)
#
# This cell:
#   1) clones the original SAT repo
#   2) applies the PyTorch 2.x batch_first compat patch on sat/models.py
#   3) keeps the GAT slot in sat/gnn_layers.py (harmless leftover from Ext-1)
#   4) writes two new modules: sat/chunking.py and sat/cross_attention.py
#   5) patches sat/data.py     to precompute chunks per graph
#   6) patches sat/models.py   to call the hierarchical block after the encoder
#   7) patches experiments/train_zinc.py with --use-chunks / --chunk-size / --max-chunks
# =============================================================================

import os, shutil, subprocess, base64

REPO = '/kaggle/working/SAT'
if os.path.isdir(REPO):
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--quiet',
                'https://github.com/BorgwardtLab/SAT.git', REPO], check=True)


# ----- 2) PyTorch 2.x batch_first patch on sat/models.py ---------------------
models_path = f'{REPO}/sat/models.py'
with open(models_path) as f:
    code = f.read()
if 'batch_first = False' not in code:
    code = code.replace(
        'self.encoder = GraphTransformerEncoder(encoder_layer, num_layers)',
        'encoder_layer.self_attn.batch_first = False\n'
        '        self.encoder = GraphTransformerEncoder(encoder_layer, num_layers)'
    )
    with open(models_path, 'w') as f:
        f.write(code)


# ----- 3) Keep the GAT slot in gnn_layers (harmless) -------------------------
gnn_path = f'{REPO}/sat/gnn_layers.py'
with open(gnn_path) as f:
    code = f.read()
if "'gat'" not in code:
    code = code.replace("'rwgnn', 'khopgnn'", "'rwgnn', 'khopgnn', 'gat'")
    code = code.replace(
        '    elif gnn_type == "gin":',
        '    elif gnn_type == "gat":\n'
        '        return gnn.GATConv(embed_dim, embed_dim // 8, heads=8)\n'
        '    elif gnn_type == "gin":'
    )
    with open(gnn_path, 'w') as f:
        f.write(code)


# ----- 4) Write the two new modules from base64 blobs -----------------------
CHUNKING_B64 = '''IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0KIiIiCkhpZXJhcmNoaWNhbCBjaHVua2luZyB1dGlsaXRpZXMgZm9yIEhpZXJhcmNoaWNhbC1TQVQuCgpQYXJ0aXRpb25zIGEgZ3JhcGggaW50byBiYWxhbmNlZCBjaHVua3MgdmlhIGdyZWVkeSBCRlMgYW5kIGJ1aWxkcyBhCmNodW5rLWxldmVsIG1hY3JvLWdyYXBoIHdob3NlIGVkZ2Ugd2VpZ2h0cyBjb3VudCB0aGUgbnVtYmVyIG9mIG9yaWdpbmFsCmVkZ2VzIGNyb3NzaW5nIGVhY2ggY2h1bmsgYm91bmRhcnkuCgpXZSB1c2UgYSBzZWxmLWNvbnRhaW5lZCBCRlMgcGFydGl0aW9uZXIgaW5zdGVhZCBvZiBNRVRJUyB0byBhdm9pZCBhbgpleHRlcm5hbCBuYXRpdmUgZGVwZW5kZW5jeS4gRm9yIFpJTkMtc2NhbGUgZ3JhcGhzIChOIDw9IDUwKSB0aGlzIGlzCmJvdGggZmFzdCBhbmQgYWRlcXVhdGUuCiIiIgppbXBvcnQgbWF0aAppbXBvcnQgdG9yY2gKCgpkZWYgX2FkamFjZW5jeV9saXN0KGVkZ2VfaW5kZXgsIG51bV9ub2Rlcyk6CiAgICBhZGogPSBbW10gZm9yIF8gaW4gcmFuZ2UobnVtX25vZGVzKV0KICAgIHNyYyA9IGVkZ2VfaW5kZXhbMF0udG9saXN0KCkKICAgIGRzdCA9IGVkZ2VfaW5kZXhbMV0udG9saXN0KCkKICAgIGZvciBzLCBkIGluIHppcChzcmMsIGRzdCk6CiAgICAgICAgYWRqW3NdLmFwcGVuZChkKQogICAgcmV0dXJuIGFkagoKCmRlZiBiYWxhbmNlZF9iZnNfcGFydGl0aW9uKGVkZ2VfaW5kZXgsIG51bV9ub2RlcywgbnVtX3BhcnRzKToKICAgICIiIkdyZWVkeSBCRlMtYmFzZWQgYmFsYW5jZWQgZ3JhcGggcGFydGl0aW9uaW5nLiIiIgogICAgaWYgbnVtX3BhcnRzIDw9IDEgb3IgbnVtX25vZGVzIDw9IG51bV9wYXJ0czoKICAgICAgICByZXR1cm4gdG9yY2guemVyb3MobnVtX25vZGVzLCBkdHlwZT10b3JjaC5sb25nKQoKICAgIHRhcmdldF9zaXplID0gbWF0aC5jZWlsKG51bV9ub2RlcyAvIG51bV9wYXJ0cykKICAgIGNodW5rX2lkID0gdG9yY2guZnVsbCgobnVtX25vZGVzLCksIC0xLCBkdHlwZT10b3JjaC5sb25nKQoKICAgIGFkaiA9IF9hZGphY2VuY3lfbGlzdChlZGdlX2luZGV4LCBudW1fbm9kZXMpCiAgICBkZWcgPSB0b3JjaC50ZW5zb3IoW2xlbihhKSBmb3IgYSBpbiBhZGpdLCBkdHlwZT10b3JjaC5sb25nKQoKICAgIGZvciBjIGluIHJhbmdlKG51bV9wYXJ0cyk6CiAgICAgICAgdW5hc3NpZ25lZCA9IChjaHVua19pZCA9PSAtMSkubm9uemVybyhhc190dXBsZT1GYWxzZSkuc3F1ZWV6ZSgtMSkKICAgICAgICBpZiB1bmFzc2lnbmVkLm51bWVsKCkgPT0gMDoKICAgICAgICAgICAgYnJlYWsKICAgICAgICBzZWVkID0gdW5hc3NpZ25lZFtkZWdbdW5hc3NpZ25lZF0uYXJnbWF4KCldLml0ZW0oKQoKICAgICAgICBxdWV1ZSA9IFtzZWVkXQogICAgICAgIGNodW5rX2lkW3NlZWRdID0gYwogICAgICAgIGNvdW50ID0gMQogICAgICAgIGhlYWQgPSAwCiAgICAgICAgd2hpbGUgaGVhZCA8IGxlbihxdWV1ZSkgYW5kIGNvdW50IDwgdGFyZ2V0X3NpemU6CiAgICAgICAgICAgIHUgPSBxdWV1ZVtoZWFkXQogICAgICAgICAgICBoZWFkICs9IDEKICAgICAgICAgICAgZm9yIHYgaW4gYWRqW3VdOgogICAgICAgICAgICAgICAgaWYgY2h1bmtfaWRbdl0gPT0gLTE6CiAgICAgICAgICAgICAgICAgICAgY2h1bmtfaWRbdl0gPSBjCiAgICAgICAgICAgICAgICAgICAgY291bnQgKz0gMQogICAgICAgICAgICAgICAgICAgIHF1ZXVlLmFwcGVuZCh2KQogICAgICAgICAgICAgICAgICAgIGlmIGNvdW50ID49IHRhcmdldF9zaXplOgogICAgICAgICAgICAgICAgICAgICAgICBicmVhawoKICAgIGNodW5rX2lkW2NodW5rX2lkID09IC0xXSA9IG51bV9wYXJ0cyAtIDEKICAgIHJldHVybiBjaHVua19pZAoKCmRlZiBidWlsZF9tYWNyb19ncmFwaChlZGdlX2luZGV4LCBjaHVua19pZCwgbnVtX2NodW5rcyk6CiAgICAiIiJFZGdlIHdlaWdodCAoYSwgYikgPSBudW1iZXIgb2Ygb3JpZ2luYWwgZWRnZXMgY3Jvc3NpbmcgYmV0d2VlbiBjaHVua3MgYSBhbmQgYi4iIiIKICAgIHNyYywgZHN0ID0gZWRnZV9pbmRleFswXSwgZWRnZV9pbmRleFsxXQogICAgc3JjX2MgPSBjaHVua19pZFtzcmNdCiAgICBkc3RfYyA9IGNodW5rX2lkW2RzdF0KICAgIG1hc2sgPSBzcmNfYyAhPSBkc3RfYwoKICAgIGlmIG5vdCBtYXNrLmFueSgpOgogICAgICAgIHJldHVybiAodG9yY2guemVyb3MoKDIsIDApLCBkdHlwZT10b3JjaC5sb25nKSwKICAgICAgICAgICAgICAgIHRvcmNoLnplcm9zKCgwLCksIGR0eXBlPXRvcmNoLmZsb2F0KSkKCiAgICBjcm9zc19zcmMgPSBzcmNfY1ttYXNrXQogICAgY3Jvc3NfZHN0ID0gZHN0X2NbbWFza10KICAgIHBhaXJfa2V5ID0gY3Jvc3Nfc3JjICogbnVtX2NodW5rcyArIGNyb3NzX2RzdAogICAgdW5pcXVlX2tleXMsIGNvdW50cyA9IHRvcmNoLnVuaXF1ZShwYWlyX2tleSwgcmV0dXJuX2NvdW50cz1UcnVlKQogICAgbWFjcm9fc3JjID0gdW5pcXVlX2tleXMgLy8gbnVtX2NodW5rcwogICAgbWFjcm9fZHN0ID0gdW5pcXVlX2tleXMgJSBudW1fY2h1bmtzCiAgICBtYWNyb19lZGdlX2luZGV4ID0gdG9yY2guc3RhY2soW21hY3JvX3NyYywgbWFjcm9fZHN0XSwgZGltPTApLmxvbmcoKQogICAgbWFjcm9fZWRnZV93ZWlnaHQgPSBjb3VudHMuZmxvYXQoKQogICAgcmV0dXJuIG1hY3JvX2VkZ2VfaW5kZXgsIG1hY3JvX2VkZ2Vfd2VpZ2h0CgoKZGVmIGNvbXB1dGVfY2h1bmtzKGVkZ2VfaW5kZXgsIG51bV9ub2RlcywgY2h1bmtfc2l6ZSwgbWF4X2NodW5rcyk6CiAgICAiIiJFbmQtdG8tZW5kIGNodW5rICsgbWFjcm8tZ3JhcGggY29tcHV0YXRpb24gZm9yIG9uZSBncmFwaC4iIiIKICAgIG51bV9jaHVua3MgPSBtYXgoMiwgbWluKG1heF9jaHVua3MsIG1hdGguY2VpbChudW1fbm9kZXMgLyBjaHVua19zaXplKSkpCiAgICBudW1fY2h1bmtzID0gbWluKG51bV9jaHVua3MsIG51bV9ub2RlcykKCiAgICBjaHVua19pZCA9IGJhbGFuY2VkX2Jmc19wYXJ0aXRpb24oZWRnZV9pbmRleCwgbnVtX25vZGVzLCBudW1fY2h1bmtzKQogICAgbWFjcm9fZWRnZV9pbmRleCwgbWFjcm9fZWRnZV93ZWlnaHQgPSBidWlsZF9tYWNyb19ncmFwaCgKICAgICAgICBlZGdlX2luZGV4LCBjaHVua19pZCwgbnVtX2NodW5rcwogICAgKQogICAgcmV0dXJuIGNodW5rX2lkLCBtYWNyb19lZGdlX2luZGV4LCBtYWNyb19lZGdlX3dlaWdodCwgbnVtX2NodW5rcwo='''
CROSS_ATTN_B64 = '''IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0KIiIiCkhpZXJhcmNoaWNhbCBjcm9zcy1hdHRlbnRpb24gYmxvY2sgKFN0YWdlIDIgKyBTdGFnZSAzIG9mIEhpZXJhcmNoaWNhbC1TQVQpLgoKU3RhZ2UgMjogTWFjcm8tR0NOIHByb3BhZ2F0ZXMgaW5mb3JtYXRpb24gYW1vbmcgY2h1bmstbGV2ZWwgZW1iZWRkaW5ncy4KU3RhZ2UgMzogRWFjaCBub2RlIGNyb3NzLWF0dGVuZHMgdG8gYWxsIGNodW5rcyB3aXRoaW4gaXRzIG93biBncmFwaCwKICAgICAgICAgdGhlbiBhIGxlYXJuYWJsZSBnYXRlIGZ1c2VzIGxvY2FsIG5vZGUgZmVhdHVyZXMgd2l0aCB0aGUKICAgICAgICAgcmV0cmlldmVkIGdsb2JhbCBjaHVuay1sZXZlbCBjb250ZXh0LgoKVGhpcyBtb2R1bGUgaXMgaW5zZXJ0ZWQgQUZURVIgdGhlIFNBVCBlbmNvZGVyIGFuZCBCRUZPUkUgZ3JhcGgtbGV2ZWwgcG9vbGluZy4KIiIiCmltcG9ydCBtYXRoCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCmZyb20gdG9yY2hfZ2VvbWV0cmljLm5uIGltcG9ydCBHQ05Db252CmZyb20gdG9yY2hfc2NhdHRlciBpbXBvcnQgc2NhdHRlcl9tZWFuLCBzY2F0dGVyCgoKY2xhc3MgSGllcmFyY2hpY2FsQ3Jvc3NBdHRlbnRpb24obm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkX21vZGVsKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmRfbW9kZWwgPSBkX21vZGVsCiAgICAgICAgc2VsZi5zY2FsZSA9IDEuMCAvIG1hdGguc3FydChkX21vZGVsKQoKICAgICAgICAjIFN0YWdlIDIgLS0gb25lIG1hY3JvLUdDTiBsYXllciB3aXRoIGVkZ2Ugd2VpZ2h0cwogICAgICAgIHNlbGYubWFjcm9fZ2NuID0gR0NOQ29udigKICAgICAgICAgICAgZF9tb2RlbCwgZF9tb2RlbCwKICAgICAgICAgICAgYWRkX3NlbGZfbG9vcHM9VHJ1ZSwgbm9ybWFsaXplPVRydWUKICAgICAgICApCgogICAgICAgICMgU3RhZ2UgMyAtLSBzaW5nbGUtaGVhZCBjcm9zcy1hdHRlbnRpb24gcHJvamVjdGlvbnMgKGtlcHQgc2ltcGxlKQogICAgICAgIHNlbGYuV19RID0gbm4uTGluZWFyKGRfbW9kZWwsIGRfbW9kZWwsIGJpYXM9RmFsc2UpCiAgICAgICAgc2VsZi5XX0sgPSBubi5MaW5lYXIoZF9tb2RlbCwgZF9tb2RlbCwgYmlhcz1GYWxzZSkKICAgICAgICBzZWxmLldfViA9IG5uLkxpbmVhcihkX21vZGVsLCBkX21vZGVsLCBiaWFzPUZhbHNlKQoKICAgICAgICAjIEdhdGVkIGZ1c2lvbiAobWlycm9ycyB0aGUgRXh0ZW5zaW9uLTMgcmVzaWR1YWwgZ2F0ZSBwYXR0ZXJuKQogICAgICAgIHNlbGYuV19nYXRlID0gbm4uTGluZWFyKDIgKiBkX21vZGVsLCBkX21vZGVsKQoKICAgICAgICAjIEZpbmFsIEJOLCBjb25zaXN0ZW50IHdpdGggdGhlIFNBVCBlbmNvZGVyJ3MgQk4gY2hvaWNlCiAgICAgICAgc2VsZi5ub3JtID0gbm4uQmF0Y2hOb3JtMWQoZF9tb2RlbCkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4LCBjaHVua19pZCwgY2h1bmtfZWRnZV9pbmRleCwgY2h1bmtfZWRnZV93ZWlnaHQsCiAgICAgICAgICAgICAgICBub2RlX2JhdGNoKToKICAgICAgICAjIC0tLS0gU3RhZ2UgMiA6IGNodW5rIGluaXQgdmlhIG1lYW4gb3ZlciBhc3NpZ25lZCBub2RlcyAtLS0tCiAgICAgICAgWiA9IHNjYXR0ZXJfbWVhbih4LCBjaHVua19pZCwgZGltPTApICAgICAgICAgICAgICAgICAgICAgICAjIFtDLCBkXQogICAgICAgIGNodW5rX2JhdGNoID0gc2NhdHRlcihub2RlX2JhdGNoLCBjaHVua19pZCwgZGltPTAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJlZHVjZT0nbWF4JywgZGltX3NpemU9Wi5zaXplKDApKSAgICAjIFtDXQoKICAgICAgICBpZiBjaHVua19lZGdlX2luZGV4Lm51bWVsKCkgPiAwOgogICAgICAgICAgICBaID0gRi5yZWx1KHNlbGYubWFjcm9fZ2NuKFosIGNodW5rX2VkZ2VfaW5kZXgsIGNodW5rX2VkZ2Vfd2VpZ2h0KSkKICAgICAgICBlbHNlOgogICAgICAgICAgICAjIE5vIGludGVyLWNodW5rIGVkZ2VzIGluIHRoZSBlbnRpcmUgYmF0Y2ggLS0gcmVseSBvbiBzZWxmLWxvb3BzCiAgICAgICAgICAgIFogPSBGLnJlbHUoc2VsZi5tYWNyb19nY24oWiwgY2h1bmtfZWRnZV9pbmRleCkpCgogICAgICAgICMgLS0tLSBTdGFnZSAzIDogbm9kZS10by1jaHVuayBjcm9zcy1hdHRlbnRpb24gLS0tLQogICAgICAgIFEgPSBzZWxmLldfUSh4KSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBbTiwgZF0KICAgICAgICBLID0gc2VsZi5XX0soWikgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgW0MsIGRdCiAgICAgICAgViA9IHNlbGYuV19WKFopICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIFtDLCBkXQoKICAgICAgICAjIEVhY2ggbm9kZSBhdHRlbmRzIE9OTFkgdG8gY2h1bmtzIHdpdGhpbiBpdHMgb3duIGdyYXBoCiAgICAgICAgc2FtZV9ncmFwaCA9IG5vZGVfYmF0Y2gudW5zcXVlZXplKDEpID09IGNodW5rX2JhdGNoLnVuc3F1ZWV6ZSgwKQogICAgICAgIHNjb3JlcyA9IChRIEAgSy50KCkpICogc2VsZi5zY2FsZSAgICAgICAgICAgICAgICAgICAgICAgICAgIyBbTiwgQ10KICAgICAgICBzY29yZXMgPSBzY29yZXMubWFza2VkX2ZpbGwofnNhbWVfZ3JhcGgsIGZsb2F0KCctaW5mJykpCiAgICAgICAgYXR0biA9IEYuc29mdG1heChzY29yZXMsIGRpbT0tMSkgICAgICAgICAgICAgICAgICAgICAgICAgICAjIFtOLCBDXQogICAgICAgIEcgPSBhdHRuIEAgViAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBbTiwgZF0KCiAgICAgICAgIyAtLS0tIEdhdGVkIGZ1c2lvbiAtLS0tCiAgICAgICAgZ2F0ZSA9IHRvcmNoLnNpZ21vaWQoc2VsZi5XX2dhdGUodG9yY2guY2F0KFt4LCBHXSwgZGltPS0xKSkpCiAgICAgICAgeF9vdXQgPSBnYXRlICogRyArICgxLjAgLSBnYXRlKSAqIHgKICAgICAgICB4X291dCA9IHNlbGYubm9ybSh4X291dCkKICAgICAgICByZXR1cm4geF9vdXQK'''

with open(f'{REPO}/sat/chunking.py', 'wb') as f:
    f.write(base64.b64decode(CHUNKING_B64))
with open(f'{REPO}/sat/cross_attention.py', 'wb') as f:
    f.write(base64.b64decode(CROSS_ATTN_B64))


# ----- 5) Patch sat/data.py --------------------------------------------------
data_path = f'{REPO}/sat/data.py'
with open(data_path) as f:
    code = f.read()

# 5a) Update my_inc to handle chunk_id and chunk_edge_index (must come BEFORE
#     the generic 'index in key' rule so that chunk_edge_index doesn't get
#     incremented by num_nodes).
old_inc = (
    "def my_inc(self, key, value, *args, **kwargs):\n"
    "    if key == 'subgraph_edge_index':\n"
    "        return self.num_subgraph_nodes \n"
    "    if key == 'subgraph_node_idx':\n"
    "        return self.num_nodes \n"
    "    if key == 'subgraph_indicator':\n"
    "        return self.num_nodes \n"
    "    elif 'index' in key:\n"
    "        return self.num_nodes\n"
    "    else:\n"
    "        return 0"
)
new_inc = (
    "def my_inc(self, key, value, *args, **kwargs):\n"
    "    if key == 'subgraph_edge_index':\n"
    "        return self.num_subgraph_nodes \n"
    "    if key == 'subgraph_node_idx':\n"
    "        return self.num_nodes \n"
    "    if key == 'subgraph_indicator':\n"
    "        return self.num_nodes \n"
    "    if key == 'chunk_edge_index':\n"
    "        return self.num_chunks\n"
    "    if key == 'chunk_id':\n"
    "        return self.num_chunks\n"
    "    elif 'index' in key:\n"
    "        return self.num_nodes\n"
    "    else:\n"
    "        return 0"
)
assert old_inc in code, 'data.py: my_inc anchor not found'
code = code.replace(old_inc, new_inc)

# 5b) Extend GraphDataset.__init__ signature with chunk args
old_sig = (
    '    def __init__(self, dataset, degree=False, k_hop=2, se="gnn", use_subgraph_edge_attr=False,\n'
    '                 cache_path=None, return_complete_index=True):'
)
new_sig = (
    '    def __init__(self, dataset, degree=False, k_hop=2, se="gnn", use_subgraph_edge_attr=False,\n'
    '                 cache_path=None, return_complete_index=True,\n'
    '                 use_chunks=False, chunk_size=5, max_chunks=16):'
)
assert old_sig in code, 'data.py: __init__ signature anchor not found'
code = code.replace(old_sig, new_sig)

# 5c) After the khopgnn block, set up chunks and add a _compute_chunks method
old_khop = (
    "        if self.se == 'khopgnn':\n"
    '            Data.__inc__ = my_inc\n'
    '            self.extract_subgraphs()'
)
new_khop = (
    "        if self.se == 'khopgnn':\n"
    '            Data.__inc__ = my_inc\n'
    '            self.extract_subgraphs()\n'
    '\n'
    '        self.use_chunks = use_chunks\n'
    '        self.chunk_size = chunk_size\n'
    '        self.max_chunks = max_chunks\n'
    '        if self.use_chunks:\n'
    '            Data.__inc__ = my_inc\n'
    '            self._compute_chunks()\n'
    '\n'
    '    def _compute_chunks(self):\n'
    '        from .chunking import compute_chunks\n'
    "        print('Computing chunk partitions (size={}, max={})...'.format(\n"
    '            self.chunk_size, self.max_chunks))\n'
    '        self.chunk_data_list = []\n'
    '        for i in range(len(self.dataset)):\n'
    '            g = self.dataset[i]\n'
    '            cid, mei, mew, nc = compute_chunks(\n'
    '                g.edge_index, int(g.num_nodes),\n'
    '                self.chunk_size, self.max_chunks\n'
    '            )\n'
    '            self.chunk_data_list.append((cid, mei, mew, nc))\n'
    "        print('Done!')"
)
assert old_khop in code, 'data.py: khopgnn anchor not found'
code = code.replace(old_khop, new_khop)

# 5d) In __getitem__, attach chunk fields after the existing else-branch
old_else = (
    '        else:\n'
    '            data.num_subgraph_nodes = None\n'
    '            data.subgraph_node_idx = None\n'
    '            data.subgraph_edge_index = None\n'
    '            data.subgraph_indicator = None\n'
    '\n'
    '        return data'
)
new_else = (
    '        else:\n'
    '            data.num_subgraph_nodes = None\n'
    '            data.subgraph_node_idx = None\n'
    '            data.subgraph_edge_index = None\n'
    '            data.subgraph_indicator = None\n'
    '\n'
    "        if getattr(self, 'use_chunks', False):\n"
    '            cid, mei, mew, nc = self.chunk_data_list[index]\n'
    '            data.chunk_id = cid\n'
    '            data.chunk_edge_index = mei\n'
    '            data.chunk_edge_weight = mew\n'
    '            data.num_chunks = int(nc)\n'
    '\n'
    '        return data'
)
assert old_else in code, 'data.py: __getitem__ anchor not found'
code = code.replace(old_else, new_else)

with open(data_path, 'w') as f:
    f.write(code)


# ----- 6) Patch sat/models.py ------------------------------------------------
with open(models_path) as f:
    code = f.read()

# 6a) Add use_chunks to GraphTransformer.__init__ signature
old_gt_sig = (
    '    def __init__(self, in_size, num_class, d_model, num_heads=8,\n'
    '                 dim_feedforward=512, dropout=0.0, num_layers=4,\n'
    '                 batch_norm=False, abs_pe=False, abs_pe_dim=0,\n'
    '                 gnn_type="graph", se="gnn", use_edge_attr=False, num_edge_features=4,\n'
    '                 in_embed=True, edge_embed=True, use_global_pool=True, max_seq_len=None,\n'
    "                 global_pool='mean', **kwargs):"
)
new_gt_sig = (
    '    def __init__(self, in_size, num_class, d_model, num_heads=8,\n'
    '                 dim_feedforward=512, dropout=0.0, num_layers=4,\n'
    '                 batch_norm=False, abs_pe=False, abs_pe_dim=0,\n'
    '                 gnn_type="graph", se="gnn", use_edge_attr=False, num_edge_features=4,\n'
    '                 in_embed=True, edge_embed=True, use_global_pool=True, max_seq_len=None,\n'
    "                 global_pool='mean', use_chunks=False, **kwargs):"
)
assert old_gt_sig in code, 'models.py: GraphTransformer signature anchor not found'
code = code.replace(old_gt_sig, new_gt_sig)

# 6b) Instantiate the hierarchical block right after the encoder is built
old_enc = (
    '        encoder_layer.self_attn.batch_first = False\n'
    '        self.encoder = GraphTransformerEncoder(encoder_layer, num_layers)\n'
    '        self.global_pool = global_pool'
)
new_enc = (
    '        encoder_layer.self_attn.batch_first = False\n'
    '        self.encoder = GraphTransformerEncoder(encoder_layer, num_layers)\n'
    '\n'
    '        # Hierarchical-SAT extension : chunk-level cross-attention block\n'
    '        self.use_chunks = use_chunks\n'
    '        if use_chunks:\n'
    '            from .cross_attention import HierarchicalCrossAttention\n'
    '            self.hier_block = HierarchicalCrossAttention(d_model)\n'
    '\n'
    '        self.global_pool = global_pool'
)
assert old_enc in code, 'models.py: encoder construction anchor not found'
code = code.replace(old_enc, new_enc)

# 6c) Call the hierarchical block after self.encoder(...) and before pooling
old_readout = (
    '            ptr=data.ptr,\n'
    '            return_attn=return_attn\n'
    '        )\n'
    '        # readout step\n'
    '        if self.use_global_pool:'
)
new_readout = (
    '            ptr=data.ptr,\n'
    '            return_attn=return_attn\n'
    '        )\n'
    '        # Hierarchical-SAT : inject chunk-level global context\n'
    '        if self.use_chunks:\n'
    '            output = self.hier_block(\n'
    '                output, data.chunk_id, data.chunk_edge_index,\n'
    '                data.chunk_edge_weight, data.batch\n'
    '            )\n'
    '        # readout step\n'
    '        if self.use_global_pool:'
)
assert old_readout in code, 'models.py: readout anchor not found'
code = code.replace(old_readout, new_readout)

with open(models_path, 'w') as f:
    f.write(code)


# ----- 7) Patch experiments/train_zinc.py ------------------------------------
train_path = f'{REPO}/experiments/train_zinc.py'
with open(train_path) as f:
    code = f.read()

# 7a) Add three new CLI flags right after --se
old_se = (
    "    parser.add_argument('--se', type=str, default=\"gnn\", \n"
    "            help='Extractor type: khopgnn, or gnn')"
)
new_se = (
    "    parser.add_argument('--se', type=str, default=\"gnn\", \n"
    "            help='Extractor type: khopgnn, or gnn')\n"
    "    parser.add_argument('--use-chunks', action='store_true',\n"
    "            help='Enable Hierarchical-SAT chunk-level cross-attention block')\n"
    "    parser.add_argument('--chunk-size', type=int, default=5,\n"
    "            help='Target nodes per chunk (C = max(2, ceil(N / chunk_size)))')\n"
    "    parser.add_argument('--max-chunks', type=int, default=16,\n"
    "            help='Hard cap on number of chunks per graph')"
)
assert old_se in code, 'train_zinc.py: --se anchor not found'
code = code.replace(old_se, new_se)

# 7b) Pass chunk args into all three GraphDataset constructions
#     (single replace -- all three call sites share the same closing kwarg)
old_ds_kw = 'use_subgraph_edge_attr=args.use_edge_attr)'
new_ds_kw = (
    'use_subgraph_edge_attr=args.use_edge_attr,\n'
    '        use_chunks=args.use_chunks, chunk_size=args.chunk_size,\n'
    '        max_chunks=args.max_chunks)'
)
assert old_ds_kw in code, 'train_zinc.py: GraphDataset closing kwarg anchor not found'
code = code.replace(old_ds_kw, new_ds_kw)

# 7c) Pass use_chunks into the GraphTransformer construction
old_gt_call_tail = (
    '                             se=args.se,\n'
    '                             deg=deg,\n'
    '                             global_pool=args.global_pool) '
)
new_gt_call_tail = (
    '                             se=args.se,\n'
    '                             deg=deg,\n'
    '                             use_chunks=args.use_chunks,\n'
    '                             global_pool=args.global_pool) '
)
assert old_gt_call_tail in code, 'train_zinc.py: GraphTransformer call anchor not found'
code = code.replace(old_gt_call_tail, new_gt_call_tail)

with open(train_path, 'w') as f:
    f.write(code)


print('All patches applied. Repo ready at', REPO)


In [ ]:
!pip install torch_geometric ogb performer-pytorch einops -q
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.10.0+cu128.html -q


In [ ]:
import sys
if '/kaggle/working/SAT' not in sys.path:
    sys.path.insert(0, '/kaggle/working/SAT')

!cd /kaggle/working/SAT && PYTHONPATH=/kaggle/working/SAT python experiments/train_zinc.py \
    --seed 0 \
    --gnn-type mpnn \
    --use-edge-attr \
    --k-hop 3 \
    --se gnn \
    --abs-pe rw \
    --abs-pe-dim 20 \
    --epochs 2000 \
    --batch-size 128 \
    --lr 0.001 \
    --weight-decay 1e-5 \
    --dropout 0.2 \
    --num-layers 6 \
    --num-heads 8 \
    --dim-hidden 64 \
    --use-chunks \
    --chunk-size 5 \
    --max-chunks 16 \
    --outdir /kaggle/working/results_hier_sat_mpnn
